# AI use at work: analysis and Tableau exports

I compare AI use with output per hour, reported time saving and daily stress across 31 countries, then repeat the comparison within 17 broadly representative Ipsos adult samples. I also examine Claude automation and descriptive trust/time-saving groups.

This publication notebook works from the included country tables. It can be run without the large raw files. Notebook 01 documents and rebuilds the source preparation. The calculations follow my final working analysis, with full precision retained in the exported statistics instead of rounding before export.

**Unit:** country. **Method:** two-sided Spearman correlation. **Main correction:** Holm across nine tests within each sample, alpha 0.05. The added automation comparison is raw and exploratory. These are associations, not causal effects.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Run from this project folder, or set AI_CAPSTONE_PROJECT_DIR to its location.
PROJECT_ROOT = Path(os.environ.get("AI_CAPSTONE_PROJECT_DIR", str(Path.cwd()))).resolve()
assert (PROJECT_ROOT / "data_clean" / "survey_values.csv").exists(), (
    "Set PROJECT_ROOT to the project folder containing data_clean/survey_values.csv."
)
CLEAN = PROJECT_ROOT / "data_clean"
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print("Project:", PROJECT_ROOT.name)

Project: 04_AI_at_Work_Tableau_Google_Colab


## 1. Check the saved country table

The representative flag is a survey-coverage sensitivity, not an income group. The map has its own fixed 121-country scope. The column gallup_stress_2025 retains its original filename-style label, but describes employee daily stress pooled over 2023–2025.

In [2]:
deep = pd.read_csv(CLEAN / "analysis_depth.csv", keep_default_na=False)
rep = deep["ipsos_representative"].astype(str).str.lower().eq("true")
assert len(deep) == 31 and deep["iso3"].is_unique
assert rep.sum() == 17
assert not deep.isna().any().any()
samples = [("All 31 countries", deep), ("General population only (17)", deep[rep])]
print("All countries:",len(deep),"| Representative subset:",int(rep.sum()))
print("Productivity years:", sorted(deep["output_year"].unique()))

All countries: 31 | Representative subset: 17
Productivity years: [np.int64(2025)]


## 2. Compare adoption and work outcomes

Three adoption measures × three outcomes produce nine tests for each sample. Holm adjustment is applied separately to the nine raw p-values in each sample. Rho describes direction/strength; p is not the probability that an explanation is true. The p-values use SciPy's standard approximation; a small-sample permutation check would be a future extension.

In [3]:
def holm(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    adjusted = np.maximum.accumulate(pvalues[order] * np.arange(len(pvalues), 0, -1))
    result = np.empty_like(pvalues)
    result[order] = np.minimum(adjusted, 1)
    return result

adoption = {"Claude per person":"claude_per_capita_index",
            "OpenAI use":"openai_use_score", "Microsoft adoption":"ms_adoption_pct"}
outcomes = {"Time saved (Ipsos)":"ipsos_time_saved_pct",
            "Daily stress (Gallup)":"gallup_stress_2025", "Output per hour (ILO)":"output_per_hour"}

def correlations(frame, xs, ys, sample, family):
    rows = []
    for xn, x in xs.items():
        for yn, y in ys.items():
            pair = frame[[x,y]].apply(pd.to_numeric).dropna()
            rho, p = spearmanr(pair[x], pair[y], alternative="two-sided")
            rows.append(dict(adoption=xn, outcome=yn, sample=sample, n=len(pair),
                             rho=float(rho), p_raw=float(p), family=family))
    result = pd.DataFrame(rows)
    result["p_holm"] = holm(result["p_raw"])
    result["significant_after_holm"] = result["p_holm"] < 0.05
    return result

main = pd.concat([correlations(frame, adoption, outcomes, label, "9 adoption/outcome tests within sample")
                  for label,frame in samples], ignore_index=True)
main.to_csv(RESULTS / "main_correlations.csv", index=False)
# Keep the original lookup column names expected by Tableau.
lookup = main[["adoption","outcome","sample","n","rho","p_raw","p_holm"]].rename(columns={"p_raw":"p"})
lookup.to_csv(CLEAN / "stats_lookup.csv", index=False)
print(main[["sample","adoption","outcome","rho","p_holm"]].round(4).to_string(index=False))

                      sample           adoption               outcome     rho  p_holm
            All 31 countries  Claude per person    Time saved (Ipsos) -0.5844  0.0033
            All 31 countries  Claude per person Daily stress (Gallup)  0.4321  0.0456
            All 31 countries  Claude per person Output per hour (ILO)  0.8043  0.0000
            All 31 countries         OpenAI use    Time saved (Ipsos) -0.5619  0.0050
            All 31 countries         OpenAI use Daily stress (Gallup)  0.1964  0.3664
            All 31 countries         OpenAI use Output per hour (ILO)  0.7911  0.0000
            All 31 countries Microsoft adoption    Time saved (Ipsos) -0.5456  0.0060
            All 31 countries Microsoft adoption Daily stress (Gallup)  0.2455  0.3664
            All 31 countries Microsoft adoption Output per hour (ILO)  0.8161  0.0000
General population only (17)  Claude per person    Time saved (Ipsos) -0.0700  1.0000
General population only (17)  Claude per person Daily 

## 3. Check the added Claude use/automation relationship

These two comparisons were added for dashboard 3 and are labelled raw/exploratory, not folded into the main nine-test family. They describe the share of chats classified as automation, not job replacement.

In [4]:
auto_rows = []
for label, frame in samples:
    rho, p = spearmanr(frame["claude_per_capita_index"], frame["claude_automation_pct"], alternative="two-sided")
    auto_rows.append(dict(sample=label, n=len(frame), rho=float(rho), p_raw=float(p),
                          correction="None; added exploratory comparison"))
automation = pd.DataFrame(auto_rows)
automation.to_csv(RESULTS / "automation_correlations.csv", index=False)
print(automation.round(6).to_string(index=False))

                      sample  n       rho    p_raw                         correction
            All 31 countries 31 -0.654567 0.000065 None; added exploratory comparison
General population only (17) 17 -0.348253 0.170733 None; added exploratory comparison


## 4. Calculate the dashboard indicators

Each KPI is an unweighted mean of country percentages. It is not a pooled estimate for all survey respondents or workers. The OpenAI doing fraction is multiplied by 100 for display as a percentage.

In [5]:
kpi_rows = []
for label,frame in samples:
    metrics = {"Trust without checking":frame["ipsos_trust_pct"],
               "Reported time saving":frame["ipsos_time_saved_pct"],
               "Claude automation":frame["claude_automation_pct"],
               "ChatGPT doing":frame["openai_work_doing_share"] * 100}
    for metric,values in metrics.items():
        kpi_rows.append(dict(sample=label,n=len(frame),metric=metric,mean_pct=float(values.mean())))
kpis = pd.DataFrame(kpi_rows)
kpis.to_csv(RESULTS / "kpi_summary.csv", index=False)
print(kpis.round(2).to_string(index=False))

                      sample  n                 metric  mean_pct
            All 31 countries 31 Trust without checking     30.52
            All 31 countries 31   Reported time saving     59.68
            All 31 countries 31      Claude automation     47.73
            All 31 countries 31          ChatGPT doing     45.54
General population only (17) 17 Trust without checking     23.82
General population only (17) 17   Reported time saving     50.35
General population only (17) 17      Claude automation     46.24
General population only (17) 17          ChatGPT doing     42.89


## 5. Compare trust and time saving across the two country groups

The dumbbell splits the active sample at its median estimated Claude task-time reduction. Values at or above the median form the higher-estimate group. These groups use a different variable from automation share. The plotted values are median country survey percentages; no test of the group difference is performed.

The legacy Tableau export calls the groups Delegate more / Delegate less. Those names are retained for compatibility, while the dashboard explains the split using estimated time saved. The export also retains the original ChatGPT split as supporting work; the published dumbbell uses the Claude estimated-time split.

In [6]:
group_rows, profile_rows = [], []
for label, frame in samples:
    threshold = float(frame["claude_est_time_saved_pct"].median())
    higher = frame["claude_est_time_saved_pct"] >= threshold
    for group, selected in [("Higher estimate",higher),("Lower estimate",~higher)]:
        g = frame[selected]
        group_rows.append(dict(sample=label,group=group,n=len(g),threshold_pct=threshold,
                               time_saved_median_pct=float(g["ipsos_time_saved_pct"].median()),
                               trust_median_pct=float(g["ipsos_trust_pct"].median())))
    for split_name, values in [("Claude: estimated task time saved",frame["claude_est_time_saved_pct"]),
                               ("ChatGPT: doing share of work",frame["openai_work_doing_share"]*100)]:
        high = values >= values.median()
        for group, selected in [("Delegate more",high),("Delegate less",~high)]:
            g = frame[selected]
            for measure,col in [("Workers who say AI saved them time (Ipsos)","ipsos_time_saved_pct"),
                                ("Trust AI without checking (Ipsos)","ipsos_trust_pct"),
                                ("Claude use per person (index)","claude_per_capita_index")]:
                profile_rows.append(dict(split_by=split_name,sample=label,group=group,
                                          n_countries=len(g),measure=measure,median=float(g[col].median())))
groups = pd.DataFrame(group_rows)
groups.to_csv(RESULTS / "group_medians.csv", index=False)
pd.DataFrame(profile_rows).to_csv(CLEAN / "delegation_profiles.csv",index=False)
print(groups.round(2).to_string(index=False))

                      sample           group  n  threshold_pct  time_saved_median_pct  trust_median_pct
            All 31 countries Higher estimate 16          85.17                   70.0              37.5
            All 31 countries  Lower estimate 15          85.17                   48.0              21.0
General population only (17) Higher estimate  9          84.05                   56.0              27.0
General population only (17)  Lower estimate  8          84.05                   45.5              20.5


## 6. Retain the exploratory depth and trust checks

These are supporting analyses from the working notebook. They use separate correction families, not one global correction for the whole project: 18 depth/outcome tests per sample and 10 trust tests per sample. They are kept separate from the main adoption comparisons.

In [7]:
depth_x = {"Claude: work share":"claude_work_pct", "Claude: automation share":"claude_automation_pct",
           "Claude: autonomy":"claude_autonomy", "Claude: est. task time saved":"claude_est_time_saved_pct",
           "ChatGPT: work share":"openai_work_share", "ChatGPT: doing share of work":"openai_work_doing_share"}
depth_results = pd.concat([correlations(frame,depth_x,outcomes,label,"18 depth/outcome tests within sample")
                           for label,frame in samples],ignore_index=True)
trust_x = {**depth_x,**adoption,"Time saved (Ipsos)":"ipsos_time_saved_pct"}
trust_results = pd.concat([correlations(frame,trust_x,{"Trust AI without checking (Ipsos)":"ipsos_trust_pct"},
                                       label,"10 trust tests within sample") for label,frame in samples],ignore_index=True)
depth_results.to_csv(RESULTS / "depth_outcome_correlations.csv",index=False)
trust_results.to_csv(RESULTS / "trust_correlations.csv",index=False)
print("Depth/outcome rows:",len(depth_results),"| Trust rows:",len(trust_results))
selected_depth = depth_results[(depth_results["sample"].str.contains("17")) &
                               depth_results["adoption"].eq("Claude: est. task time saved") &
                               depth_results["outcome"].eq("Time saved (Ipsos)")]
print(selected_depth[["rho","p_raw","p_holm"]].round(6).to_string(index=False))

Depth/outcome rows: 36 | Trust rows: 20
     rho  p_raw   p_holm
0.804176 0.0001 0.001797


## 7. Save the fixed matching counts and record the environment

The full matching path stays fixed at 121 → 121 → 119 → 31 → 31. Gallup and Ipsos both contribute to the selected 31-country comparison; these are this study's retained counts, not complete provider coverage. Tableau displays the fixed matching history even when the sample changes to 17.

In [8]:
funnel = pd.DataFrame({"stage":["Anthropic","+ OpenAI","+ ILOSTAT","+ Gallup","+ Ipsos (final)"],
                       "countries":[121,121,119,31,31]})
funnel.to_csv(CLEAN / "funnel.csv",index=False)
assert len(main) == 18 and len(automation) == 2
assert not main[main["sample"].str.contains("17")]["significant_after_holm"].any()
import platform
import scipy
print("Python:",platform.python_version())
print("pandas:",pd.__version__,"| NumPy:",np.__version__,"| SciPy:",scipy.__version__)
print("Saved statistics, summaries and Tableau lookup/group exports.")
print("These files do not refresh or republish the live Tableau workbook automatically.")

Python: 3.12.14
pandas: 3.0.1 | NumPy: 2.3.5 | SciPy: 1.18.1
Saved statistics, summaries and Tableau lookup/group exports.
These files do not refresh or republish the live Tableau workbook automatically.


## What I found

The full comparison has strong positive productivity associations and negative reported-time-saving associations. The main results weaken in the 17-country subset; none of its nine tests passes Holm. Stress has no robust, consistent association across the checks. Estimated task-time reduction and automation are different measures and should not be used interchangeably.

The README describes the published dashboards, sources and limitations. Small rounding differences from the historical Tableau lookup come from keeping full precision in this publication edition; they do not represent a new cohort or a new causal finding. See the raw versus adjusted p columns and their family labels before quoting a result.